# MLP Baseline with Shared MFCC Cache


## 1. Package Setup


In [ ]:
# Purpose: Package installs are intentionally not run automatically.\n# %pip install tensorflow scikit-learn pandas numpy matplotlib seaborn


## 2. Load MLP-Ready Cache


In [ ]:
import json
import os
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.models import Sequential

try:
    from IPython.display import display
except Exception:
    display = print

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
tf.keras.utils.set_random_seed(RANDOM_STATE)
os.environ["PYTHONHASHSEED"] = str(RANDOM_STATE)

MAX_EPOCHS = 50
EARLY_STOP_PATIENCE = 7
EARLY_STOP_MIN_DELTA = 1e-4
THRESHOLD = 0.5
RUN_BASELINE_TRAINING = False
CLASS_NAMES = {0: "bona_fide", 1: "synthetic"}


def resolve_project_root():
    explicit = os.environ.get("INTRO_AI_PROJECT_ROOT")
    if explicit:
        return Path(explicit).expanduser().resolve()
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "Model Variants").exists():
            return candidate
        if (candidate / "Training" / "Model Variants").exists():
            return candidate / "Training"
    return Path("/content/drive/MyDrive/Colab Notebooks/Education/INM701")


PROJECT_ROOT = resolve_project_root()
SHARED_CLASS_WEIGHT_PATH = PROJECT_ROOT / "outputs" / "shared" / "class_weights.json"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "mlp"
TABLES_DIR = OUTPUT_DIR / "tables"
CACHE_DIR = OUTPUT_DIR / "cache"
HPO_DIR = OUTPUT_DIR / "hpo"
MODELS_DIR = OUTPUT_DIR / "models"
METRICS_DIR = OUTPUT_DIR / "metrics"
FIGURES_DIR = OUTPUT_DIR / "figures"
for directory in [OUTPUT_DIR, TABLES_DIR, CACHE_DIR, HPO_DIR, MODELS_DIR, METRICS_DIR, FIGURES_DIR]:
    directory.mkdir(parents=True, exist_ok=True)


def require_file(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(
            f"Required file not found: {path}. Run the analysis notebook, then 00_MLP_Data_Preparation.ipynb."
        )
    return path


def load_shared_class_weights(path):
    with open(require_file(path), "r", encoding="utf-8") as f:
        payload = json.load(f)
    weights = payload.get("class_weights", payload)
    return {int(label): float(weight) for label, weight in weights.items()}


for required in [
    CACHE_DIR / "X_train.npy",
    CACHE_DIR / "y_train.npy",
    CACHE_DIR / "train_metadata.csv",
    CACHE_DIR / "X_validation.npy",
    CACHE_DIR / "y_validation.npy",
    CACHE_DIR / "validation_metadata.csv",
    CACHE_DIR / "feature_config.json",
    SHARED_CLASS_WEIGHT_PATH,
]:
    require_file(required)

X_train = np.load(CACHE_DIR / "X_train.npy")
y_train = np.load(CACHE_DIR / "y_train.npy")
train_metadata = pd.read_csv(CACHE_DIR / "train_metadata.csv")
X_validation = np.load(CACHE_DIR / "X_validation.npy")
y_validation = np.load(CACHE_DIR / "y_validation.npy")
validation_metadata = pd.read_csv(CACHE_DIR / "validation_metadata.csv")
with open(CACHE_DIR / "feature_config.json", "r", encoding="utf-8") as f:
    feature_config = json.load(f)
class_weights = load_shared_class_weights(SHARED_CLASS_WEIGHT_PATH)

print("Loaded MLP-ready cache:", CACHE_DIR)
print("X_train shape:", X_train.shape)
print("X_validation shape:", X_validation.shape)
print("Shared class weights:", class_weights)


## 3. Fixed Baseline Validation Run


In [ ]:
BASELINE_CONFIG = {
    "hidden_units": [128, 64],
    "activation": "relu",
    "dropout": 0.20,
    "learning_rate": 0.001,
    "batch_size": 128,
}


def build_mlp_model(config, input_dim):
    model = Sequential()
    model.add(Input(shape=(input_dim,)))
    for units in config["hidden_units"]:
        model.add(Dense(units, activation=config["activation"]))
        model.add(Dropout(config["dropout"]))
    model.add(Dense(1, activation="sigmoid"))
    optimizer = tf.keras.optimizers.Adam(learning_rate=config["learning_rate"])
    model.compile(loss="binary_crossentropy", optimizer=optimizer, metrics=["accuracy"])
    return model


if RUN_BASELINE_TRAINING:
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(RANDOM_STATE)
    model = build_mlp_model(BASELINE_CONFIG, X_train.shape[1])
    model.summary()

    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=EARLY_STOP_PATIENCE,
            min_delta=EARLY_STOP_MIN_DELTA,
            restore_best_weights=True,
        ),
        tf.keras.callbacks.ModelCheckpoint(
            str(MODELS_DIR / "mlp_baseline_fixed.keras"),
            monitor="val_loss",
            save_best_only=True,
        ),
    ]

    start_time = time.perf_counter()
    history = model.fit(
        X_train,
        y_train,
        validation_data=(X_validation, y_validation),
        epochs=MAX_EPOCHS,
        batch_size=BASELINE_CONFIG["batch_size"],
        class_weight=class_weights,
        callbacks=callbacks,
        verbose=2,
    )
    runtime_seconds = time.perf_counter() - start_time

    validation_probability = model.predict(X_validation, batch_size=BASELINE_CONFIG["batch_size"], verbose=0).ravel()
    validation_pred = (validation_probability >= THRESHOLD).astype(int)
    baseline_results = {
        "method": "baseline_fixed_mlp",
        "validation_macro_f1": f1_score(y_validation, validation_pred, average="macro", zero_division=0),
        "validation_binary_f1_synthetic": f1_score(y_validation, validation_pred, pos_label=1, zero_division=0),
        "validation_accuracy": accuracy_score(y_validation, validation_pred),
        "validation_precision_synthetic": precision_score(y_validation, validation_pred, pos_label=1, zero_division=0),
        "validation_recall_synthetic": recall_score(y_validation, validation_pred, pos_label=1, zero_division=0),
        "best_val_loss": float(np.min(history.history["val_loss"])),
        "best_epoch": int(np.argmin(history.history["val_loss"]) + 1),
        "epochs_trained": int(len(history.history["loss"])),
        "runtime_seconds": float(runtime_seconds),
        "configuration": json.dumps(BASELINE_CONFIG, sort_keys=True),
    }
    pd.DataFrame([baseline_results]).to_csv(HPO_DIR / "baseline_validation_metrics.csv", index=False)
    with open(HPO_DIR / "baseline_training_history.json", "w", encoding="utf-8") as f:
        json.dump(history.history, f, indent=2)
    display(pd.DataFrame([baseline_results]))
else:
    print("RUN_BASELINE_TRAINING is False. Set it to True when ready to train the MLP baseline.")


## 4. Reproducibility Checklist


In [ ]:
checks = {
    "uses_shared_canonical_split": True,
    "loads_mlp_ready_cache": str(CACHE_DIR),
    "does_not_rescan_audio": True,
    "does_not_create_split": True,
    "does_not_extract_mfcc": True,
    "input_representation": "80-D MFCC mean/std vector",
    "uses_shared_class_weights": str(SHARED_CLASS_WEIGHT_PATH),
    "test_set_used_here": False,
}
display(pd.DataFrame(list(checks.items()), columns=["check", "value"]))
